# Cybersecurity Incident Response Under Uncertainty

**University Assignment — Experiment 6**

This notebook reproduces the experiment's calculations and results, including Bayesian updating, belief states, expected utility, Value of Information (VoI), MDP/POMDP, BDK-POMDP, the 540-run simulation, Brier score, and Incident Response Confidence Score (MGS).

**Experiment settings:** random seed = 42, repetitions = 5, gamma = 0.90, maximum permitted steps = 5.

> **Note:** The experiment documentation does not specify a numerical delay-penalty coefficient. Therefore, the delay penalty is set to 0 in the single-step expected-utility calculation. Delay is evaluated separately in sequential models and in the OMG metric.


In [ ]:
from pathlib import Path
import importlib.util, csv, json
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

BASE = Path("../csv")
print('Working directory:', BASE)


## 1. Run the Main Experiment Code

In [ ]:
spec = importlib.util.spec_from_file_location('deney6_solution', BASE/'deney6_solution.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
print('Main simulation completed.')


## 2. Bayesian Calculation for Observation O1

In [ ]:
bayes_o1 = pd.read_csv(BASE/'bayes_o1.csv')
bayes_o1


## 3. Final Belief State After Sequential Observations

In [ ]:
belief = pd.read_csv(BASE/'belief_updates.csv')
final_belief = belief.groupby('Senaryo', as_index=False).tail(1)
final_belief[['Senaryo','P(S0)','P(S1)','P(S2)','P(S3)','En_Olasi_Durum']].reset_index(drop=True)


## 4. Expected Utility and Value of Information

In [ ]:
eu = pd.read_csv(BASE/'expected_utility.csv')
best_eu = eu.loc[eu.groupby('Senaryo')['Beklenen_Fayda'].idxmax(), ['Senaryo','Eylem','Beklenen_Fayda']]
best_eu.reset_index(drop=True)


In [ ]:
voi = pd.read_csv(BASE/'value_of_information.csv')
voi[voi['Bilgi_Degeri'] > 0][['Senaryo','Test','Bilgi_Degeri','Karar']].reset_index(drop=True)


## 5. MDP Policy

In [ ]:
pd.read_csv(BASE/'mdp_policy.csv')


## 6. 540 Experimental Runs and Overall Metrics

In [ ]:
runs = pd.read_csv(BASE/'experiment_540_runs.csv')
print('Total experimental runs:', len(runs))
metrics = pd.read_csv(BASE/'general_metrics.csv')
metrics


In [ ]:
plt.figure(figsize=(8,4))
plt.bar(metrics['Model'], metrics['MGS'])
plt.ylim(0,100)
plt.xlabel('Model')
plt.ylabel('MGS')
plt.title('Incident Response Confidence Score by Model')
plt.show()


## 7. Ablation Analysis

In [ ]:
ablation = pd.read_csv(BASE/'ablation_results.csv')
ablation


In [ ]:
plt.figure(figsize=(7,4))
plt.bar(ablation['Yapilandirma'], ablation['MGS'])
plt.ylim(0,100)
plt.xlabel('Configuration')
plt.ylabel('MGS')
plt.title('Ablation Analysis')
plt.show()


## 8. Conclusion

- After observation O1, state S1 has the highest posterior probability.
- Fixed-threshold and raw-LLM approaches may produce false isolation decisions in normal scenarios.
- M4 is an oracle MDP that is assumed to know the true state; therefore, it represents a theoretical upper bound.
- Among the applicable models that do not know the true state, M8 (LLM + BDK-POMDP control) achieves the highest MGS value.
- The Value of Information gate evaluates the net decision benefit of an additional test in uncertain cases such as S6, helping avoid unnecessary tests.
